In [18]:
# Cell 1: Import
import os, time, copy, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, auc)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("models", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("Device:", DEVICE)

Device: cpu


In [19]:
# Cell 2: run_epoch — dùng chung cho train/val/test

def run_epoch(model, loader, criterion, optimizer=None, is_binary=False):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, ys, ps, probs_all = 0.0, [], [], []
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb)
            if is_binary:
                out = out.squeeze(-1)
                loss = criterion(out, yb.float())
                probs = torch.sigmoid(out)
                preds = (probs >= 0.5).long()
            else:
                loss = criterion(out, yb)
                probs = torch.softmax(out, dim=1)
                preds = out.argmax(dim=1)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            ys.append(yb.cpu()); ps.append(preds.cpu())
            probs_all.append(probs.detach().cpu())
    y_true = torch.cat(ys).numpy(); y_pred = torch.cat(ps).numpy()
    y_prob = torch.cat(probs_all).numpy()
    return {
        'loss': total_loss / len(y_true),
        'acc':  accuracy_score(y_true, y_pred),
        'f1':   f1_score(y_true, y_pred, average='macro', zero_division=0),
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
    }

In [20]:
# Cell 3: train_model — pipeline huấn luyện dùng chung

def train_model(model, loaders, epochs=15, lr=1e-3, wd=1e-4,
                patience=5, is_binary=False, class_weights=None,
                save_path=None, verbose=True):
    model = model.to(DEVICE)

    if is_binary and class_weights is not None:
        criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights[1:2])
    elif (not is_binary) and class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.BCEWithLogitsLoss() if is_binary else nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc','val_f1']}
    best_f1, best_state, bad = -1.0, None, 0
    t0 = time.time()

    for ep in range(1, epochs + 1):
        tr = run_epoch(model, loaders['train'], criterion, optimizer, is_binary)
        va = run_epoch(model, loaders['val'],   criterion, None,      is_binary)
        scheduler.step()

        for k, v in zip(['train_loss','val_loss','train_acc','val_acc','val_f1'],
                        [tr['loss'], va['loss'], tr['acc'], va['acc'], va['f1']]):
            history[k].append(v)

        if verbose:
            print(f"[Epoch {ep:02d}] train_loss={tr['loss']:.4f} | "
                  f"val_loss={va['loss']:.4f} | val_acc={va['acc']:.4f} | val_f1={va['f1']:.4f}")

        if va['f1'] > best_f1:
            best_f1, bad = va['f1'], 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                if verbose: print("→ Early stopping.")
                break

    train_time = time.time() - t0
    model.load_state_dict(best_state)
    test_m = run_epoch(model, loaders['test'], criterion, None, is_binary)
    test_m['params'] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    test_m['train_time'] = train_time
    test_m['history'] = history

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save({'state': best_state, 'history': history,
                    'params': test_m['params']}, save_path)
    return test_m

Task was destroyed but it is pending!
task: <Task pending name='Task-260' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\The Hoang\miniconda3\envs\ml\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-261' coro=<Kernel.shell_main() running at C:\Users\The Hoang\miniconda3\envs\ml\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\The Hoang\miniconda3\envs\ml\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\The Hoang\miniconda3\envs\ml\Lib\tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-261' coro=<Kernel.shell_main() running at C:\Users\The Hoang\miniconda3\envs\ml\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


In [21]:
# Cell 4: Vẽ learning curves
def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title(f"{title} — Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'],   label='Val')
    axes[1].set_title(f"{title} — Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    axes[2].plot(history['val_f1'], color='green', label='Val F1')
    axes[2].set_title(f"{title} — Val Macro F1"); axes[2].set_xlabel("Epoch"); axes[2].legend()
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=200)
    plt.show()

In [22]:
# Cell 5: Vẽ confusion matrix
def plot_confusion(y_true, y_pred, labels, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(title); plt.xlabel("Dự đoán"); plt.ylabel("Thực tế")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=200)
    plt.show()

In [23]:
# Cell 6: Vẽ ROC-AUC cho cả nhị phân và đa lớp

def compute_roc_auc(model, loader, n_classes=10, is_binary=False):
    model.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            p = torch.sigmoid(out.squeeze(-1)) if is_binary else torch.softmax(out, dim=1)
            probs_all.append(p.cpu().numpy()); y_all.append(yb.numpy())
    probs = np.concatenate(probs_all); y = np.concatenate(y_all)

    if is_binary:
        fpr, tpr, _ = roc_curve(y, probs)
        return fpr, tpr, auc(fpr, tpr)

    if n_classes == 2:
        probs_pos = probs[:, 1]
        fpr, tpr, _ = roc_curve(y, probs_pos)
        return fpr, tpr, auc(fpr, tpr)

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve((y == i).astype(int), probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    return fpr, tpr, roc_auc, np.mean(list(roc_auc.values()))

In [24]:
# Cell 7: Tổng hợp kết quả 12 thực nghiệm
import pandas as pd
def summarize_results(results_dict):
    rows = []
    for (dataset, model_name), r in results_dict.items():
        rows.append({
            'Dataset': dataset, 'Model': model_name,
            'Params': r['params'],
            'Train(s)': round(r['train_time'], 1),
            'Acc': round(r['acc'], 4),
            'F1':  round(r['f1'], 4),
        })
    return pd.DataFrame(rows).sort_values(['Dataset', 'F1'],
                                          ascending=[True, False])
print("Cell 7: Sẵn sàng tổng hợp kết quả.")

Cell 7: Sẵn sàng tổng hợp kết quả.
